# 🌍 Good Vibes Workshop — ECCB 2026

## Responsible use of AI for large-scale, rapid Earth Observation workflows

**Authors:** AB, HFT, HH, JW  
**Affiliation:** Institute of Zoology, Zoological Society of London  
**Licence:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

---

### How this notebook works

This is not a normal coding tutorial. **You will use an AI assistant** (Gemini, ChatGPT, Copilot — whatever you prefer) to write code for you.

Each section follows this pattern:

1. 🎯 **Goal** — what we want to achieve
2. 🤖 **Prompt** — a suggested prompt to give your AI assistant
3. 💻 **AI Output** — what the AI typically generates (pre-filled so we're all on the same page)
4. ▶️ **Run it** — see what happens (does it work? does it fail? are the results correct?)
5. 📚 **Teaching moment** — what we learned, which principle applies
6. 🔄 **Iterate** — improve with a follow-up prompt

As the notebook progresses, **we remove the scaffolding** — by the end, you're vibe coding on your own.

**Open your AI assistant in another browser tab now.** You'll need it throughout.

---
# Section 0: Setup & Authentication ⚙️

Run these cells first. If you completed the pre-workshop setup, this should just work.

In [ ]:
# Install required packages
!pip install -q geemap earthengine-api

In [ ]:
# Imports
import ee
import geemap
import datetime
import sys
import matplotlib.pyplot as plt

In [ ]:
# Authenticate with Google Earth Engine
# This will open a browser window — follow the prompts
ee.Authenticate()

In [ ]:
# ⚡ IMPORTANT: Replace 'YOUR-PROJECT-ID' with your actual Google Cloud Project ID
ee.Initialize(project='YOUR-PROJECT-ID')

In [ ]:
# Quick test — if this prints without error, you're ready!
test = ee.ImageCollection('MODIS/061/MOD13A1').filterDate('2023-01-01', '2023-02-01')
print(f'✅ Setup successful! Found {test.size().getInfo()} MODIS images.')

---
# Section 1: Reproducibility Metadata 📋

Before we start, let's capture our environment. This is **Principle 6: Maximise Transparency** in action — if someone runs this notebook next year, they'll know exactly what versions we used.

In [ ]:
# Reproducibility metadata — run this once at the start
print('=' * 50)
print('REPRODUCIBILITY METADATA')
print('=' * 50)
print(f'Date:           {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Python:         {sys.version.split(" ")[0]}')
print(f'earthengine-api: {ee.__version__}')
print(f'geemap:         {geemap.__version__}')
print(f'Hansen dataset: UMD/hansen/global_forest_change_2023_v1_11')
print('=' * 50)

---
# Worked Example 1: Your First Vibe Code ✨

**This one works.** We'll ask AI to do something common and well-documented, and it'll nail it.

---

### 🎯 Goal
Visualise satellite imagery of **Sabah, Borneo** — true-colour and false-colour composites on an interactive map.

### 🤖 Prompt 1
Copy this prompt and paste it into your AI assistant:

> *Write Python code for Google Colab that uses the earthengine-api and geemap packages to display a true-colour Landsat 8 surface reflectance composite of Sabah, Borneo (approximately 115.5–118.5°E, 4.5–7°N) for 2023. Filter clouds, create an interactive map, and add the imagery as a layer.*

### 💻 What AI typically generates
Your AI's output will look something like this. Run the cell below:

In [ ]:
# 🤖 AI-Generated Code — Prompt 1
# "Display a true-colour Landsat 8 composite of Sabah, Borneo"

# Define the study area
sabah = ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])

# Load Landsat 8 surface reflectance, filter by date and cloud cover
landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .filterBounds(sabah)
           .filterDate('2023-01-01', '2023-12-31')
           .filter(ee.Filter.lt('CLOUD_COVER', 30))
           .select(['SR_B4', 'SR_B3', 'SR_B2'])  # Red, Green, Blue
           .median())  # Cloud-free composite

# Set up the map
Map = geemap.Map()
Map.centerObject(sabah, 8)

# Visualisation parameters for true colour
vis_params = {
    'min': 7000,
    'max': 20000,
    'bands': ['SR_B4', 'SR_B3', 'SR_B2']
}

Map.addLayer(landsat, vis_params, 'True Colour (2023)')
Map.addLayer(sabah, {'color': 'yellow'}, 'Study Area', opacity=0.3)
Map

### ✅ It works!

You just visualised satellite imagery for an entire region of Borneo in ~10 seconds of prompting.

> ⚠️ **Teaching Point:** AI excels at common, well-documented tasks. Landsat + GEE + geemap is extensively documented online, so the AI has seen thousands of examples. It got the band names, date format, and visualisation parameters right.

---

### 🔄 Now let's build on it

### 🤖 Prompt 2
Give your AI this follow-up:

> *Now add a false-colour composite layer using bands SR_B5, SR_B4, SR_B3 (NIR, Red, Green) to highlight vegetation. Also add protected area boundaries from the WDPA dataset (ee.FeatureCollection('WCMC/WDPA/current/polygons')) filtered to Malaysia.*

### 💻 What AI typically generates

In [ ]:
# 🤖 AI-Generated Code — Prompt 2 (building on Prompt 1)
# "Add false-colour and protected area boundaries"

# Load Landsat 8 with NIR band included
landsat_full = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                .filterBounds(sabah)
                .filterDate('2023-01-01', '2023-12-31')
                .filter(ee.Filter.lt('CLOUD_COVER', 30))
                .select(['SR_B5', 'SR_B4', 'SR_B3', 'SR_B2'])
                .median())

# Protected areas in the region
wdpa = (ee.FeatureCollection('WCMC/WDPA/current/polygons')
        .filterBounds(sabah))

# Set up the map
Map2 = geemap.Map()
Map2.centerObject(sabah, 8)

# True colour
Map2.addLayer(landsat_full,
              {'min': 7000, 'max': 20000, 'bands': ['SR_B4', 'SR_B3', 'SR_B2']},
              'True Colour')

# False colour (vegetation appears bright red/pink)
Map2.addLayer(landsat_full,
              {'min': 7000, 'max': 30000, 'bands': ['SR_B5', 'SR_B4', 'SR_B3']},
              'False Colour (Vegetation)')

# Protected areas
Map2.addLayer(wdpa, {'color': 'cyan'}, 'Protected Areas (WDPA)')

Map2

### ✅ It works again!

Notice how **iterative prompting** lets you build complexity step by step. You didn't need to know the WDPA dataset ID or the false-colour band combination — the AI did.

> ⚠️ **Teaching Point — Principle 1: Treat AI as an intelligence.** We gave it specific context (band names, dataset ID, region). The more precise your prompt, the better the output. Compare *"show me Borneo"* with what we actually asked — the specificity matters.

> 🔧 **GEE Best Practice:** Notice the code uses `.filterBounds()` and `.filterDate()` early, and `.select()` limits the bands loaded. This reduces memory usage — a best practice that AI sometimes gets right, sometimes doesn't.

---
# Worked Example 2: When Vibe Coding Fails 💥

**This one breaks.** We'll ask AI to do something slightly more complex, and the output will fail. This is normal — and learning to handle it is the whole point.

---

### 🎯 Goal
Analyse **deforestation in Sabah** using the Hansen Global Forest Change dataset. Calculate annual forest loss and plot the results.

### 🤖 Prompt

> *Write Python code for Google Colab using earthengine-api and geemap to analyse deforestation in Sabah, Borneo (115.5–118.5°E, 4.5–7°N) using the Hansen Global Forest Change dataset. Calculate the area of forest lost each year from 2001 to 2023 in hectares, and plot the results as a bar chart using matplotlib.*

### 💻 What AI typically generates

Run this cell — **it will fail!** That's the point.

In [ ]:
# 🤖 AI-Generated Code — This WILL FAIL
# "Analyse deforestation in Sabah using Hansen Global Forest Change"

# Load Hansen Global Forest Change dataset
hansen = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')

# Define study area
sabah = ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])

# Get tree cover and loss data
tree_cover = hansen.select('tree_cover')  # ❌ BUG: wrong band name!
loss_year = hansen.select('lossyear')
loss = hansen.select('loss')

# Create forest mask (>= 30% tree cover)
forest_mask = tree_cover.gte(30)

# Calculate loss area per year
years = range(1, 24)  # 1 = 2001, 23 = 2023
annual_loss = {}

for year in years:
    year_mask = loss_year.eq(year).And(forest_mask)
    area = year_mask.multiply(ee.Image.pixelArea()).divide(10000)  # hectares
    stats = area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=sabah,
        scale=30,
        maxPixels=1e13
    )
    annual_loss[2000 + year] = stats.getInfo()['lossyear']

# Plot
plt.figure(figsize=(12, 5))
plt.bar(annual_loss.keys(), annual_loss.values(), color='darkred')
plt.xlabel('Year')
plt.ylabel('Forest Loss (hectares)')
plt.title('Annual Forest Loss in Sabah, Borneo')
plt.show()

print(f'Total forest loss: {sum(annual_loss.values()):,.0f} hectares')

### 💥 It failed!

You should see an error like:
```
EEException: Image.select: Pattern 'tree_cover' did not match any bands.
```

The AI used `'tree_cover'` but the actual band name in the Hansen dataset is `'treecover2000'` (no underscore, with year).

> ⚠️ **Teaching Point — Principle 2: Distrust.** AI confidently generates plausible-looking code with wrong details. Band names, dataset IDs, and parameter values are common failure points. **Always check against the official documentation.**

---

### 🔄 Fix it with a better prompt

### 🤖 Prompt 2 (the fix)

> *The code failed because 'tree_cover' is not a valid band name. The Hansen Global Forest Change dataset (UMD/hansen/global_forest_change_2023_v1_11) has these bands: treecover2000, loss, gain, lossyear, datamask. Fix the code and make sure it uses the correct band names.*

### 💻 The corrected code

In [ ]:
# 🤖 AI-Generated Code — FIXED version
# Key fix: 'tree_cover' → 'treecover2000'

# Load Hansen Global Forest Change dataset
hansen = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')

# Define study area
sabah = ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])

# Get tree cover and loss data — CORRECT band names
tree_cover = hansen.select('treecover2000')  # ✅ Fixed!
loss_year = hansen.select('lossyear')

# Create forest mask (>= 30% tree cover in 2000)
forest_mask = tree_cover.gte(30)

# Calculate loss area per year
years = range(1, 24)  # 1 = 2001, 23 = 2023
annual_loss = {}

for year in years:
    # Mask to this year's loss within forested areas
    year_mask = loss_year.eq(year).And(forest_mask)
    # Convert pixel count to area in hectares
    area = year_mask.multiply(ee.Image.pixelArea()).divide(10000)
    stats = area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=sabah,
        scale=30,
        maxPixels=1e13,
        bestEffort=True
    )
    annual_loss[2000 + year] = stats.getInfo()['lossyear']

# Plot
plt.figure(figsize=(12, 5))
plt.bar(annual_loss.keys(), annual_loss.values(), color='darkred')
plt.xlabel('Year')
plt.ylabel('Forest Loss (hectares)')
plt.title('Annual Forest Loss in Sabah, Borneo (2001–2023)')
plt.tight_layout()
plt.show()

print(f'\nTotal forest loss 2001–2023: {sum(annual_loss.values()):,.0f} hectares')
print(f'Average annual loss: {sum(annual_loss.values()) / 23:,.0f} hectares')
worst_year = max(annual_loss, key=annual_loss.get)
print(f'Worst year: {worst_year} ({annual_loss[worst_year]:,.0f} ha)')

### ✅ Now it works!

Notice what we did:
1. **Read the error message** — it told us exactly what was wrong
2. **Gave the AI the correct information** — the actual band names from the documentation
3. **Asked it to fix the specific problem** — not a vague "fix it", but a precise correction

> ⚠️ **Teaching Point — Principle 1: Treat AI as an intelligence.** When you give it the error message *and* the correct context, it can usually fix itself. The skill is knowing **what context to provide**.

> 🔧 **GEE Best Practice:** The fixed code uses `bestEffort=True` in `reduceRegion` — this prevents timeout errors on large regions by allowing GEE to use a coarser resolution if needed. AI doesn't always include this.

---

### 🔄 Build further — let's visualise on a map

### 🤖 Prompt 3

> *Now create a geemap map showing: (1) tree cover in 2000 in green, (2) all forest loss in red, (3) forest loss coloured by year using a cool-to-warm palette. Center on Sabah.*

### 💻 What AI generates

In [ ]:
# 🤖 AI-Generated Code — Prompt 3
# "Visualise deforestation on a map"

Map3 = geemap.Map()
Map3.centerObject(sabah, 9)

# Tree cover in 2000 (masked to >= 30%)
Map3.addLayer(
    tree_cover.updateMask(forest_mask),
    {'min': 30, 'max': 100, 'palette': ['lightgreen', 'darkgreen']},
    'Tree Cover 2000'
)

# All forest loss (binary — red)
loss_binary = hansen.select('loss').updateMask(forest_mask)
Map3.addLayer(
    loss_binary.selfMask(),
    {'palette': ['red']},
    'Forest Loss (all years)'
)

# Loss by year (colour-ramped)
loss_by_year = loss_year.updateMask(loss_year.gt(0)).updateMask(forest_mask)
Map3.addLayer(
    loss_by_year,
    {'min': 1, 'max': 23, 'palette': ['yellow', 'orange', 'red', 'darkred']},
    'Loss by Year (2001–2023)'
)

Map3

### ✅ Beautiful.

Toggle the layers on and off. Zoom into areas of heavy deforestation — you'll see the palm oil plantation grid pattern.

**We built this entire analysis in three prompts.** That's the power of vibe coding. But now let's see the danger.

---

---
# Worked Example 3: Silent Bugs — It Runs, But It's Wrong 🔇

**This is the most dangerous failure mode.** The code runs without errors, produces output that *looks* reasonable — but the results are scientifically wrong.

---

### 🎯 Goal
We asked AI to calculate summary statistics about forest cover and loss in a region. The code below runs perfectly. But **three of the results are wrong.** Can you spot them?

### 🤖 The prompt we used

> *Calculate the following for Sabah, Borneo using Hansen Global Forest Change: (1) average tree cover percentage in 2000, (2) total forest loss area 2010–2020 in hectares, (3) forest loss as a percentage of original forest area.*

### 💻 What AI generated — **this code has 3 silent bugs**

Run it. It won't crash. But the numbers are wrong. **Work in pairs — can you find all three?**

In [ ]:
# 🤖 AI-Generated Code — CONTAINS 3 SILENT BUGS
# Can you find them? The code runs, but the results are WRONG.

hansen = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')
sabah = ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])

# --- Statistic 1: Average tree cover in 2000 ---
tree_cover = hansen.select('treecover2000')
avg_tc = tree_cover.reduceRegion(
    reducer=ee.Reducer.sum(),       # 🔍 BUG #1: Can you spot it?
    geometry=sabah,
    scale=1000,                     # 🔍 BUG #2: Can you spot it?
    maxPixels=1e13,
    bestEffort=True
).getInfo()
print(f'Average tree cover (2000): {avg_tc["treecover2000"]:.1f}%')

# --- Statistic 2: Total forest loss 2010-2020 ---
loss_year = hansen.select('lossyear')
loss_2010_2020 = loss_year.gte(10).And(loss_year.lte(19))  # 🔍 BUG #3: Can you spot it?
loss_area = loss_2010_2020.multiply(ee.Image.pixelArea()).divide(10000)
total_loss = loss_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=sabah,
    scale=30,
    maxPixels=1e13,
    bestEffort=True
).getInfo()
print(f'Forest loss 2010-2020: {total_loss["lossyear"]:,.0f} hectares')

# --- Statistic 3: Loss as percentage of original forest ---
forest_mask = tree_cover.gte(30)
forest_area = forest_mask.multiply(ee.Image.pixelArea()).divide(10000)
total_forest = forest_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=sabah,
    scale=30,
    maxPixels=1e13,
    bestEffort=True
).getInfo()

pct_lost = (total_loss['lossyear'] / total_forest['treecover2000']) * 100
print(f'Original forest area: {total_forest["treecover2000"]:,.0f} hectares')
print(f'Percentage lost (2010-2020): {pct_lost:.1f}%')

### 🔍 Hints

Stuck? Here are some clues. Try to find the bugs before opening the hints.

- **Bug 1 hint:** What's the difference between a *sum* and a *mean*? What does it mean to "sum" percentages across millions of pixels?
- **Bug 2 hint:** The Hansen dataset has a native resolution of 30m. What happens when you compute statistics at a different scale?
- **Bug 3 hint:** In the Hansen dataset, `lossyear` value 10 = year 2010, value 20 = year 2020. Does the code include 2020?

In [ ]:
# @title 👁️ Click to reveal Bug #1

# BUG #1: ee.Reducer.sum() instead of ee.Reducer.mean()
#
# The code SUMS tree cover percentages across millions of pixels.
# That gives a nonsensical number in the billions.
# We want the AVERAGE tree cover percentage — use ee.Reducer.mean().
#
# Fix:
#   reducer=ee.Reducer.mean()    # not .sum()
#
# WHY AI GETS THIS WRONG: Sum is the most common reducer in GEE
# tutorials (for area calculations), so AI defaults to it even
# when mean is the correct choice.

print('Bug #1: Use ee.Reducer.mean() for averaging percentages, not ee.Reducer.sum()')

In [ ]:
# @title 👁️ Click to reveal Bug #2

# BUG #2: scale=1000 instead of scale=30
#
# The Hansen dataset has 30m resolution. Using scale=1000 means
# GEE computes statistics at 1km resolution — it averages every
# ~1,111 pixels into one. This massively distorts the results:
# - Fine-scale deforestation patterns are smoothed away
# - Edge pixels are misrepresented
# - The "average" tree cover is biased
#
# Fix:
#   scale=30    # Match the native resolution of the dataset
#
# WHY AI GETS THIS WRONG: AI often uses round numbers like 1000
# for scale because many tutorials use coarse scales for speed.
# For accurate science, match the native resolution.

print('Bug #2: Use scale=30 to match the 30m native resolution of Hansen data')

In [ ]:
# @title 👁️ Click to reveal Bug #3

# BUG #3: .lte(19) instead of .lte(20)
#
# The code says "forest loss 2010–2020" but uses:
#   loss_year.gte(10).And(loss_year.lte(19))
#
# lossyear value 20 = year 2020, so .lte(19) MISSES 2020 entirely.
# The code only captures 2010–2019 — an off-by-one error.
#
# Fix:
#   loss_year.gte(10).And(loss_year.lte(20))   # Include 2020
#
# Also: the loss_2010_2020 mask includes ALL pixels with lossyear
# >= 10, including non-forest. Should intersect with forest_mask.
#
# WHY AI GETS THIS WRONG: Off-by-one errors are extremely common
# in AI-generated code, especially with encoded year values.

print('Bug #3: .lte(20) not .lte(19) — include 2020! Also mask to forest areas.')

### 📚 The lesson

All three bugs are things that **real AI systems commonly generate**. The code ran without errors. The output looked like numbers. But:

- Bug 1 produced a number in the billions (should be 0–100%)
- Bug 2 silently degraded the accuracy by 1000x
- Bug 3 excluded an entire year of data

> ⚠️ **Teaching Point — Principle 3: Verify.** If you can't independently check whether the output is correct, you can't trust it. Always:
> - Check orders of magnitude ("is this number reasonable?")
> - Verify against known values or published data
> - Test with a small, known-answer region first

> ⚠️ **Teaching Point — Principle 5: Engineer Adversity.** Ask yourself: "What would go wrong if this parameter were different?" Change the scale, the date range, the reducer — see if the results change as expected.

---

---
# Exercise 1: You Write the Prompt ✍️

Now the scaffolding comes off. **You write the prompt. You debug the output.**

---

### 🎯 Goal
Adapt the deforestation analysis (from Worked Example 2) to analyse **your own region of interest**.

### Instructions

1. **Choose a region** you're interested in. Some suggestions:
   - 🌎 Brazilian Amazon: `[-62, -5, -58, -1]`
   - 🌍 Congo Basin: `[18, -2, 22, 2]`
   - 🌏 Sumatra: `[101, -2, 105, 2]`
   - 🌍 Madagascar: `[46, -17, 50, -13]`
   - Or find your own coordinates on Google Maps (right-click → copy coordinates)

2. **Write a prompt** for your AI assistant. Here's a template to start from:

> *Write Python code using earthengine-api and geemap to analyse deforestation in [YOUR REGION] ([YOUR COORDINATES]) using the Hansen Global Forest Change dataset (UMD/hansen/global_forest_change_2023_v1_11). The band names are: treecover2000, loss, gain, lossyear, datamask. Calculate annual forest loss in hectares from 2001 to 2023 using a tree cover threshold of 30%, and plot a bar chart. Use scale=30 and bestEffort=True.*

3. **Paste the AI's output** in the cell below and run it
4. **Debug** if it fails — check the error message and give it back to the AI
5. **Verify** the results — do the numbers make sense for your region?

> 💡 **Tip:** Notice how the prompt template includes the correct band names and scale. This is **Principle 1** — giving the AI the right context prevents the bugs we saw in Worked Example 2.

In [ ]:
# ✍️ YOUR TURN — Paste your AI-generated code here and run it




In [ ]:
# ✍️ Follow-up — need more cells? Use this one.




### ✅ Checklist for Exercise 1

Before moving on, check:
- [ ] Can you explain every line of the code your AI generated?
- [ ] Did you verify the band names are correct?
- [ ] Is the scale set to 30 (matching Hansen's native resolution)?
- [ ] Do the results make sense? (Google the deforestation rate for your region and compare)
- [ ] Did the code work on the first try, or did you need to iterate?

---

---
# Exercise 2: Full Vibe Coding Challenge 🏆

**No scaffolding. No template. Just a goal.**

---

### 🎯 Challenge

**Calculate the 5-year moving average of annual deforestation in Sabah (115.5–118.5°E, 4.5–7°N) and identify the 5-year period with the highest average annual loss.**

### Rules

- Use the Hansen dataset: `UMD/hansen/global_forest_change_2023_v1_11`
- Study area: `ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])`
- You can use AI to help — that's the point!
- But you must be able to **explain your answer** to an instructor
- **Closest answer wins a prize** 🎁

### Hints
- You'll need the annual loss data from Worked Example 2 (or regenerate it)
- A 5-year moving average means: average of years [n, n+1, n+2, n+3, n+4]
- You might want to do the GEE computation first, then the moving average in plain Python
- `numpy` has a convenient function for this, or you can write it yourself

In [ ]:
# 🏆 YOUR CHALLENGE CODE — vibe code your way to victory!




In [ ]:
# 🏆 More space for your challenge code




---
# Exercise 3: Innovate — Impress Us! 🚀

*If you've finished the challenge or want to try something different.*

---

Use AI to build something we haven't shown you. Some ideas:

- 🔥 **Fire monitoring** — use MODIS/FIRMS fire data (`FIRMS`) to map recent fire hotspots
- 🌿 **NDVI trends** — calculate vegetation greenness trends over time using `MODIS/061/MOD13A1`
- 🏙️ **Urban expansion** — map built-up area change using the Global Human Settlement Layer
- 💧 **Water body change** — detect changes in water extent using JRC Global Surface Water (`JRC/GSW1_4/GlobalSurfaceWater`)
- 🌊 **Mangrove loss** — combine Hansen data with mangrove extent maps (`CIFOR/CGMF`)
- 🌡️ **Land surface temperature** — map heat islands using `MODIS/061/MOD11A2`

**Remember the six principles!** Especially Principle 2 (Distrust) and Principle 3 (Verify).

In [ ]:
# 🚀 Your innovation here!




---
# Wrap-up 🎓

## What you learned today

| # | Principle | What we practised |
|---|-----------|------------------|
| 1 | **Treat AI as an intelligence** | Giving specific context in prompts (band names, scales, dataset IDs) |
| 2 | **Distrust** | Seeing AI confidently generate wrong band names (WE2) |
| 3 | **Verify** | Finding silent bugs in code that runs without errors (WE3) |
| 4 | **Prioritize reporting & interpretation** | Understanding every line before trusting results |
| 5 | **Engineer adversity** | Changing parameters to test robustness |
| 6 | **Maximise transparency** | Capturing reproducibility metadata (Section 1) |

## Key takeaways

1. **AI is powerful** — we built a complete deforestation analysis in minutes
2. **AI is unreliable** — band names, scales, and reducers are common failure points
3. **Silent bugs are the most dangerous** — code that runs but gives wrong answers
4. **Iterative prompting works** — start simple, build complexity, fix errors with context
5. **Your domain knowledge matters** — AI writes code, but *you* decide if the science is right

## Resources

- [Google Earth Engine Documentation](https://developers.google.com/earth-engine)
- [geemap Documentation](https://geemap.org/)
- [Hansen Global Forest Change](https://glad.earthengine.app/view/global-forest-change)
- [GEE Dataset Catalog](https://developers.google.com/earth-engine/datasets)

## Don't forget

📋 **Print the vibe coding checklist** and stick it on your monitor!

💾 **Save this notebook** to your Google Drive: File → Save a copy in Drive

---

*Good Vibes Workshop — ECCB 2026*  
*AB, HFT, HH, JW*  
*Institute of Zoology, Zoological Society of London*  
*CC BY 4.0*